# Spark / Databricks Deep Dive – hands-on

Practical companion to the *Senior Deep Dive* notes, exercised on the `car_workshop`
dataset. Seven sections, each runnable on its own:

1. **Data skewness** – detect it, see it, fix it
2. **Salting** – full, two-stage aggregation, hybrid
3. **Shuffle** – narrow vs wide, `shuffle.partitions`, join strategies
4. **Auto Loader** – schema evolution, `_rescued_data`, overwrite gotcha (lab sandbox)
5. **CDC** – MERGE upsert with dedup + out-of-order guard
6. **CDF** – reading the change feed, incremental downstream replication
7. **Delta vs Parquet** – time travel, RESTORE, OPTIMIZE on your real small files

**Prerequisites:** dims loaded, facts backfilled (the more days the better – skew and
shuffle only get interesting at scale).

Everything this notebook creates lives in the **`car_workshop.lab`** schema and the
`car_workshop.lab.files` volume – drop them when you are done (last cell).
While running sections 1–3, keep the **Spark UI open** (cluster → Spark UI → Stages)
and compare *max* vs *median* task metrics – that is where skew is visible.

> **Compute note:** the mechanics demos in sections 1–3 flip Spark configs
> (`autoBroadcastJoinThreshold`, AQE flags, `shuffle.partitions`) and touch the RDD API –
> that is blocked on serverless. Those cells carry a `CLASSIC CLUSTER ONLY` banner and each
> has a serverless-friendly variant **right below it**. Sections 4–7 run on serverless as-is.

In [1]:
import time

import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = 'car_workshop'
LAB = f'{CATALOG}.lab'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {LAB}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {LAB}.files')
LAB_DIR = f'/Volumes/{CATALOG}/lab/files'


def timed(label, fn):
    t0 = time.time()
    result = fn()
    print(f'{label}: {time.time() - t0:.1f}s')
    return result


print(f'lab schema: {LAB}, lab volume: {LAB_DIR}')

lab schema: car_workshop.lab, lab volume: /Volumes/car_workshop/lab/files


## 1. Data skewness

Mental model: partition = task, a stage finishes when its **slowest** task finishes.
Skew appears only at **wide** transformations, because `partition_id = hash(key) % numPartitions`
– and `hash(NULL)` is constant, so all NULLs land in one partition.

Your dataset has natural skew built in: ~30% of `fact_sales_transactions.customer_id`
is NULL (walk-in customers).

In [2]:
# how skewed is customer_id? (run this BEFORE any expensive join)
# good way to check what is going on with data
# NOTE: check the distribution of the key you are about to JOIN / GROUP BY on -
# that key comes from the QUERY (business question), not from the table's storage
# partitioning. Storage layout is a separate concern -> next cell.

trx = spark.table(f'{CATALOG}.fact.fact_sales_transactions')

dist = trx.groupBy('customer_id').agg(F.count('*').alias('cnt'))
stats = dist.select(
    F.max('cnt').alias('max_cnt'),
    F.expr('percentile_approx(cnt, 0.5)').alias('median_cnt'),
).first()
print(f"hottest key: {stats['max_cnt']:,} rows | median: {stats['median_cnt']:,} "
      f"| skew ratio: {stats['max_cnt'] / stats['median_cnt']:.0f}x")
print('rule of thumb: <5x fine, >20x AQE may not be enough -> broadcast / salting')

display(dist.orderBy(F.desc('cnt')).limit(10))  # the NULL row dominates

hottest key: 1,987,691 rows | median: 9 | skew ratio: 220855x
rule of thumb: <5x fine, >20x AQE may not be enough -> broadcast / salting


,customer_id,cnt
0,NaN,1987691
1,281008.0,27
2,260189.0,25
3,275260.0,25
4,25626.0,25
5,43281.0,25
6,287994.0,25
7,174158.0,24
8,40741.0,24
9,483032.0,24


In [ ]:
# Storage layout inspection - a DIFFERENT question than the shuffle-key check above:
#   shuffle skew -> distribution of the JOIN/GROUP BY key (previous cell)
#   storage skew -> distribution of rows across PARTITIONS on disk (this cell)

# 1) programmatic: DESCRIBE DETAIL exposes layout as proper columns
#    (bronze = hive partitions, silver = liquid clustering - compare!)
for table in [f'{CATALOG}.fact.fact_sales_transactions',
              f'{CATALOG}.silver.sales_transactions']:
    d = spark.sql(f'DESCRIBE DETAIL {table}').first()
    print(f"{table}\n  partitionColumns={d['partitionColumns']}  "
          f"clusteringColumns={d['clusteringColumns']}  numFiles={d['numFiles']:,}")

# 2) human-readable: full metadata incl. '# Partition Information' section
display(spark.sql(f'DESCRIBE TABLE EXTENDED {CATALOG}.fact.fact_sales_transactions'))

# 3) rows per STORAGE partition - storage skew / small-files check
#    (uneven partitions hurt writes & pruning; join skew is a different beast)
display(spark.table(f'{CATALOG}.fact.fact_sales_transactions')
        .groupBy('year', 'month').count().orderBy('year', 'month'))

car_workshop.fact.fact_sales_transactions
  partitionColumns=['year', 'month']  clusteringColumns=[]


ModuleNotFoundError: No module named 'delta.connect.proto'

In [5]:
# build an artificially skewed table for the exercises: 80% of rows -> one hot product
items = spark.table(f'{CATALOG}.fact.fact_sales_items')
hot_product = items.groupBy('product_id').count().orderBy(F.desc('count')).first()['product_id']

(items
 .withColumn('product_id',
             F.when(F.rand(seed=42) < 0.8, F.lit(hot_product)).otherwise(F.col('product_id')))
 .write.mode('overwrite').saveAsTable(f'{LAB}.skewed_sales_items'))

skewed = spark.table(f'{LAB}.skewed_sales_items')
display(skewed.groupBy('product_id').agg(F.count('*').alias('cnt')).orderBy(F.desc('cnt')).limit(5))

,product_id,cnt
0,355,15905905
1,342,8769
2,359,8674
3,191,8640
4,385,8618


In [ ]:
# =====================================================================
# CLASSIC CLUSTER ONLY (session with mentor) - FAILS on serverless:
#   - spark.conf.set(...) used below is not on the serverless allowlist
#   Serverless-friendly variant: NEXT CELL.
# =====================================================================

products = spark.table(f'{CATALOG}.dim.dim_products')

# force the BAD path: no broadcast (dim is tiny, Spark would broadcast it away),
# no AQE skew handling -> classic skewed sort-merge join
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'false')
timed('skewed SMJ, AQE skew join OFF',
      lambda: skewed.join(products, 'product_id').agg(F.sum('value_net')).collect())
# -> Spark UI: one task with huge Shuffle Read vs tiny median = the skew signature

spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
timed('same join, AQE skew join ON (splits oversized shuffle partitions)',
      lambda: skewed.join(products, 'product_id').agg(F.sum('value_net')).collect())

# the real fix here: broadcast - no shuffle by key, skew becomes irrelevant
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')
timed('broadcast join',
      lambda: skewed.join(F.broadcast(products), 'product_id').agg(F.sum('value_net')).collect())

In [ ]:
# SERVERLESS variant - the bad plan cannot be forced here: configs are blocked
# and AQE is always on. Confirm the engine dodges the skew by broadcasting the
# dim on its own, then inspect the run via the QUERY PROFILE (serverless
# replacement for the classic Spark UI Stages view).
skewed_df = spark.table(f'{LAB}.skewed_sales_items')
products_df = spark.table(f'{CATALOG}.dim.dim_products')

skewed_df.join(products_df, 'product_id').agg(F.sum('value_net')).explain()

timed('engine-managed join on serverless',
      lambda: skewed_df.join(products_df, 'product_id').agg(F.sum('value_net')).collect())

In [ ]:
# =====================================================================
# CLASSIC CLUSTER ONLY (session with mentor) - FAILS on serverless:
#   - spark.conf.set(...) used below is not on the serverless allowlist
#   Serverless-friendly variant: NEXT CELL.
# =====================================================================

# NULL-skew fix: give NULL keys a random NON-matching value before a left join.
# dim customer_ids are positive -> negatives never match, left-join semantics preserved,
# but the "NULL pile" spreads across many partitions.
customers = spark.table(f'{CATALOG}.dim.dim_customers')

trx_nullsafe = trx.withColumn(
    'customer_id',
    F.when(F.col('customer_id').isNull(),
           (F.rand(seed=1) * -1_000_000).cast('bigint') - 1)
     .otherwise(F.col('customer_id')))

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)  # force SMJ to see the effect
timed('left join, NULLs as-is   ', lambda: trx.join(customers, 'customer_id', 'left').count())
timed('left join, NULLs scattered', lambda: trx_nullsafe.join(customers, 'customer_id', 'left').count())
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')

In [ ]:
# SERVERLESS variant - the NULL-scatter transformation is plain DataFrame code:
# practice the technique and verify the left-join semantics are preserved.
# (The timing difference only shows with broadcast disabled -> classic cluster.)
trx_df = spark.table(f'{CATALOG}.fact.fact_sales_transactions')
customers_df = spark.table(f'{CATALOG}.dim.dim_customers')

trx_nullsafe_df = trx_df.withColumn(
    'customer_id',
    F.when(F.col('customer_id').isNull(),
           (F.rand(seed=1) * -1_000_000).cast('bigint') - 1)
     .otherwise(F.col('customer_id')))

before = trx_df.join(customers_df, 'customer_id', 'left').count()
after = trx_nullsafe_df.join(customers_df, 'customer_id', 'left').count()
print(f'left-join semantics preserved: {before == after} ({before:,} rows)')

## 2. Salting

Manually raise the cardinality of a hot key: `"PL"` becomes `"PL_0"..."PL_15"`.
Price: the other side must be **replicated ×SALT**, otherwise `PL_7` finds no partner.
Use only when AQE skew join + broadcast are not enough (single mega-hot key,
dimension too big to broadcast, deterministic control needed).

In [ ]:
# =====================================================================
# CLASSIC CLUSTER ONLY (session with mentor) - FAILS on serverless:
#   - spark.conf.set(...) used below is not on the serverless allowlist
#   Serverless-friendly variant: NEXT CELL.
# =====================================================================

SALT = 16
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)      # forbid broadcast
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'false')  # forbid AQE rescue

# big skewed side: random salt per row
fact_salted = (skewed
    .withColumn('salt', (F.rand(seed=7) * SALT).cast('int'))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt')))

# small side: replicate EVERY row to all salt values
salt_range = spark.range(SALT).withColumnRenamed('id', 'salt')
dim_salted = (products.crossJoin(salt_range)
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt'))
    .drop('product_id', 'salt'))

timed('salted sort-merge join (hot key spread over 16 partitions)',
      lambda: fact_salted.join(dim_salted, 'k_salt').agg(F.sum('value_net')).collect())

spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')

In [ ]:
# SERVERLESS variant - salting is plain DataFrame code, so exercise CORRECTNESS:
# the salted join must return exactly the same aggregate as the plain join
# (this is the part people get wrong - forgetting the dim replication).
# The performance effect needs a forced SMJ -> classic cluster.
SALT = 16
skewed_df = spark.table(f'{LAB}.skewed_sales_items')
products_df = spark.table(f'{CATALOG}.dim.dim_products')

fact_salted_sl = (skewed_df
    .withColumn('salt', (F.rand(seed=7) * SALT).cast('int'))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt')))

dim_salted_sl = (products_df
    .crossJoin(spark.range(SALT).withColumnRenamed('id', 'salt'))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt'))
    .drop('product_id', 'salt'))

plain = skewed_df.join(products_df, 'product_id').agg(F.round(F.sum('value_net'), 2)).first()[0]
salted = fact_salted_sl.join(dim_salted_sl, 'k_salt').agg(F.round(F.sum('value_net'), 2)).first()[0]
print(f'plain join:  {plain:,}')
print(f'salted join: {salted:,}  -> identical: {plain == salted}')

In [ ]:
# two-stage (salted) AGGREGATION - for distributive aggs: sum/count/min/max
# stage 1: partial agg on (key, salt) -> stage 2: final agg on key
partial = (skewed
    .withColumn('salt', (F.rand(seed=7) * SALT).cast('int'))
    .groupBy('product_id', 'salt')
    .agg(F.sum('value_net').alias('partial_sum'), F.count('*').alias('partial_cnt')))

final = (partial.groupBy('product_id')
    .agg(F.sum('partial_sum').alias('revenue'),
         (F.sum('partial_sum') / F.sum('partial_cnt')).alias('avg_value')))  # avg = sum/count!

display(final.orderBy(F.desc('revenue')).limit(5))
# note: count(distinct) can NOT be salted this way

In [ ]:
# =====================================================================
# CLASSIC CLUSTER ONLY (session with mentor) - FAILS on serverless:
#   - spark.conf.set(...) used below is not on the serverless allowlist
#   Serverless-friendly variant: NEXT CELL.
# =====================================================================

# hybrid / isolated salting (senior-level): salt ONLY the hot keys,
# so the dimension is replicated x16 just for a handful of keys
counts = skewed.groupBy('product_id').agg(F.count('*').alias('cnt'))
median_cnt = counts.select(F.expr('percentile_approx(cnt, 0.5)')).first()[0]
hot = counts.filter(F.col('cnt') > 20 * median_cnt).select('product_id')
print(f'hot keys: {hot.count()} (threshold: 20x median = {20 * median_cnt:,} rows)')

fact2 = (skewed
    .join(F.broadcast(hot.withColumn('is_hot', F.lit(True))), 'product_id', 'left')
    .withColumn('salt', F.when(F.col('is_hot').isNotNull(),
                               (F.rand(seed=7) * SALT).cast('int')).otherwise(F.lit(0)))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt')))

dim_hot = products.join(F.broadcast(hot), 'product_id', 'inner').crossJoin(salt_range)
dim_cold = products.join(F.broadcast(hot), 'product_id', 'left_anti').withColumn('salt', F.lit(0))
dim2 = (dim_hot.unionByName(dim_cold)
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt'))
    .drop('product_id', 'salt'))

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
timed('hybrid-salted join', lambda: fact2.join(dim2, 'k_salt').agg(F.sum('value_net')).collect())
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')

In [ ]:
# SERVERLESS variant - hybrid salting without the conf toggles: verify the logic
# (row counts must match the plain join), benchmark later on the classic cluster.
SALT = 16
skewed_df = spark.table(f'{LAB}.skewed_sales_items')
products_df = spark.table(f'{CATALOG}.dim.dim_products')
salt_range_sl = spark.range(SALT).withColumnRenamed('id', 'salt')

counts_sl = skewed_df.groupBy('product_id').agg(F.count('*').alias('cnt'))
median_sl = counts_sl.select(F.expr('percentile_approx(cnt, 0.5)')).first()[0]
hot_sl = counts_sl.filter(F.col('cnt') > 20 * median_sl).select('product_id')

fact2_sl = (skewed_df
    .join(F.broadcast(hot_sl.withColumn('is_hot', F.lit(True))), 'product_id', 'left')
    .withColumn('salt', F.when(F.col('is_hot').isNotNull(),
                               (F.rand(seed=7) * SALT).cast('int')).otherwise(F.lit(0)))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt')))

dim_hot_sl = products_df.join(F.broadcast(hot_sl), 'product_id', 'inner').crossJoin(salt_range_sl)
dim_cold_sl = products_df.join(F.broadcast(hot_sl), 'product_id', 'left_anti').withColumn('salt', F.lit(0))
dim2_sl = (dim_hot_sl.unionByName(dim_cold_sl)
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt'))
    .drop('product_id', 'salt'))

plain_cnt = skewed_df.join(products_df, 'product_id').count()
hybrid_cnt = fact2_sl.join(dim2_sl, 'k_salt').count()
print(f'plain: {plain_cnt:,} rows, hybrid-salted: {hybrid_cnt:,} -> identical: {plain_cnt == hybrid_cnt}')

## 3. Shuffle

Physical redistribution of rows over the network so that rows with the same key meet
in one task. Cost = disk I/O + network + (de)serialization + a **stage barrier**.
In `explain()` output, every `Exchange` = one shuffle = one stage boundary.

In [ ]:
print('=== NARROW (filter/select) - no Exchange ===')
trx.filter(F.col('payment_method') == 'card').select('transaction_id').explain()

print('=== WIDE (groupBy) - Exchange hashpartitioning ===')
trx.groupBy('location_id').count().explain()

In [ ]:
# =====================================================================
# CLASSIC CLUSTER ONLY (session with mentor) - FAILS on serverless:
#   - spark.conf.set(...) used below is not on the serverless allowlist
#   - .rdd API is not supported on serverless
#   Serverless-friendly variant: NEXT CELL.
# =====================================================================

# spark.sql.shuffle.partitions - the most important knob (default 200)
# target ~128-256 MB per partition after shuffle
spark.conf.set('spark.sql.adaptive.enabled', 'false')   # show the raw effect first

spark.conf.set('spark.sql.shuffle.partitions', 8)
print('shuffle.partitions=8   ->', trx.groupBy('customer_id').count().rdd.getNumPartitions(), 'partitions')

spark.conf.set('spark.sql.shuffle.partitions', 400)
print('shuffle.partitions=400 ->', trx.groupBy('customer_id').count().rdd.getNumPartitions(), 'partitions')

spark.conf.set('spark.sql.adaptive.enabled', 'true')    # AQE coalesces small partitions back
print('with AQE coalesce      ->', trx.groupBy('customer_id').count().rdd.getNumPartitions(), 'partitions')
spark.conf.set('spark.sql.shuffle.partitions', 'auto')

In [ ]:
# SERVERLESS variant - shuffle partitions are engine-managed (auto AQE), but you
# can still OBSERVE how many partitions a shuffle produced - without the RDD API:
def n_partitions(df):
    """Count distinct physical partitions the rows actually landed in."""
    return (df.withColumn('_pid', F.spark_partition_id())
              .select(F.countDistinct('_pid')).first()[0])


trx_df = spark.table(f'{CATALOG}.fact.fact_sales_transactions')
print('partitions after groupBy shuffle (engine-chosen):',
      n_partitions(trx_df.groupBy('customer_id').count()))
# manually tuning 8 vs 400 vs AQE coalesce -> classic cluster

In [ ]:
# =====================================================================
# CLASSIC CLUSTER ONLY (session with mentor) - FAILS on serverless:
#   - spark.conf.set(...) used below is not on the serverless allowlist
#   - .rdd API is not supported on serverless
#   Serverless-friendly variant: NEXT CELL.
# =====================================================================

# join strategies: read them from the plan
locations = spark.table(f'{CATALOG}.dim.dim_locations')

print('=== dim below autoBroadcastJoinThreshold -> BroadcastHashJoin (no shuffle) ===')
trx.join(locations, 'location_id').explain()

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
print('=== broadcast disabled -> SortMergeJoin (shuffle BOTH sides) ===')
trx.join(locations, 'location_id').explain()
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')

# repartition vs coalesce:
sample = trx.limit(1_000_000)
print('repartition(50) ->', sample.repartition(50).rdd.getNumPartitions(), '(full shuffle, round-robin)')
print('coalesce(4)     ->', sample.coalesce(4).rdd.getNumPartitions(), '(narrow merge, no shuffle)')
# repartition("key") hashes by key again -> re-creates skew for hot keys!

In [ ]:
# SERVERLESS variant - you cannot forbid broadcast, but plans are still readable
# and partition counts observable (n_partitions defined in the previous cell):
trx_df = spark.table(f'{CATALOG}.fact.fact_sales_transactions')
locations_df = spark.table(f'{CATALOG}.dim.dim_locations')

print('=== engine picks BroadcastHashJoin on its own - find it in the plan ===')
trx_df.join(locations_df, 'location_id').explain()

sample_sl = trx_df.limit(1_000_000)
print('repartition(50) ->', n_partitions(sample_sl.repartition(50)), '(full shuffle, round-robin)')
print('coalesce(4)     ->', n_partitions(sample_sl.coalesce(4)), '(narrow merge, no shuffle)')

## 4. Auto Loader

Production ingest lives in `autoloader.ipynb` – here a **sandbox** on the lab volume
to trigger the behaviours you get quizzed on: schema inference, `_rescued_data`,
and the overwrite gotcha. Two separate state locations: `schemaLocation`
(inferred/evolving schema) and `checkpointLocation` (which files are done).

In [ ]:
import json as pyjson

LANDING = f'{LAB_DIR}/landing/orders'
SCHEMA_LOC = f'{LAB_DIR}/_schemas/orders'
CHK_ORDERS = f'{LAB_DIR}/_checkpoints/orders'

# clean start (lab volume only - safe)
dbutils.fs.rm(f'{LAB_DIR}/landing', True)
dbutils.fs.rm(SCHEMA_LOC, True)
dbutils.fs.rm(CHK_ORDERS, True)
spark.sql(f'DROP TABLE IF EXISTS {LAB}.orders_bronze')

batch1 = [{'order_id': i, 'amount': round(10 + i * 1.5, 2)} for i in range(1, 6)]
dbutils.fs.put(f'{LANDING}/batch1.json', '\n'.join(pyjson.dumps(r) for r in batch1), True)


def ingest_orders():
    (spark.readStream.format('cloudFiles')
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.schemaLocation', SCHEMA_LOC)
        .option('cloudFiles.schemaEvolutionMode', 'rescue')   # never fail: misfits -> _rescued_data
        .option('cloudFiles.schemaHints', 'order_id BIGINT, amount DOUBLE')
        .load(LANDING)
        .writeStream
        .option('checkpointLocation', CHK_ORDERS)
        .trigger(availableNow=True)
        .toTable(f'{LAB}.orders_bronze')
        .awaitTermination())


ingest_orders()
display(spark.table(f'{LAB}.orders_bronze'))

In [ ]:
# batch 2: a NEW column and a BROKEN type -> with rescue mode nothing fails,
# misfits land in _rescued_data as JSON. Monitor it: non-null = source changed format!
batch2 = [
    {'order_id': 6, 'amount': 99.9, 'currency': 'PLN'},   # unexpected column
    {'order_id': 'oops-a-string', 'amount': 12.3},        # type mismatch
]
dbutils.fs.put(f'{LANDING}/batch2.json', '\n'.join(pyjson.dumps(r) for r in batch2), True)

ingest_orders()
display(spark.table(f'{LAB}.orders_bronze').orderBy(F.col('order_id').asc_nulls_last()))

In [ ]:
# the overwrite gotcha: Auto Loader tracks files BY NAME - overwriting an already
# processed file is silently IGNORED (classic "why is my new data missing")
batch1_fixed = [{'order_id': i, 'amount': 999.99} for i in range(1, 6)]
dbutils.fs.put(f'{LANDING}/batch1.json', '\n'.join(pyjson.dumps(r) for r in batch1_fixed), True)

before = spark.table(f'{LAB}.orders_bronze').count()
ingest_orders()
after = spark.table(f'{LAB}.orders_bronze').count()
print(f'rows before: {before}, after re-ingest of overwritten file: {after} (no change!)')
print('fix: cloudFiles.allowOverwrites=true, or write new files instead of overwriting')
# related options: cloudFiles.backfillInterval (safety net for missed notifications),
# cloudFiles.maxFilesPerTrigger / maxBytesPerTrigger (rate limiting)

## 5. CDC (pattern, not a feature)

CDC = capturing inserts/updates/deletes from a source and shipping only the delta.
Below: a simulated change feed for customers -> **MERGE upsert** with the three details
interviews ask about:

1. **dedup** – newest change per key inside a batch (MERGE errors on multiple matches),
2. **out-of-order guard** – `s.op_ts > t.op_ts` so a late event cannot overwrite newer state,
3. **idempotency** – re-running the batch is a no-op.

In [ ]:
# build the change feed: snapshot + updates + LATE updates + deletes
spark.sql(f'DROP TABLE IF EXISTS {LAB}.customer_changes')
spark.sql(f'DROP TABLE IF EXISTS {LAB}.customers_current')
dbutils.fs.rm(f'{LAB_DIR}/_checkpoints/customers_current', True)

base = (spark.table(f'{CATALOG}.dim.dim_customers')
        .select('customer_id', 'first_name', 'last_name', 'city', 'email')
        .filter('customer_id <= 1000'))


def as_changes(df, op, ts):
    return df.withColumn('op', F.lit(op)).withColumn('op_ts', F.lit(ts).cast('timestamp'))


changes = (
    as_changes(base, 'i', '2026-01-01 00:00:00')                              # initial snapshot
    .union(as_changes(base.filter('customer_id <= 100')
                      .withColumn('city', F.lit('Warszawa-NEW')), 'u', '2026-01-03 12:00:00'))
    .union(as_changes(base.filter('customer_id <= 30')                        # LATE: older ts
                      .withColumn('city', F.lit('LATE-SHOULD-LOSE')), 'u', '2026-01-02 09:00:00'))
    .union(as_changes(base.filter('customer_id BETWEEN 901 AND 950'), 'd', '2026-01-03 13:00:00'))
)
changes.write.mode('overwrite').saveAsTable(f'{LAB}.customer_changes')

spark.sql(f"""
    CREATE TABLE {LAB}.customers_current (
        customer_id BIGINT, first_name STRING, last_name STRING,
        city STRING, email STRING, op_ts TIMESTAMP)
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)   -- needed for section 6
""")
print(f'change feed: {changes.count()} rows')

In [ ]:
def upsert_customers(batch_df, batch_id):
    # 1) dedup: keep the NEWEST change per key within this batch
    latest = (batch_df
              .withColumn('rn', F.row_number().over(
                  Window.partitionBy('customer_id').orderBy(F.col('op_ts').desc())))
              .filter('rn = 1').drop('rn'))

    (DeltaTable.forName(spark, f'{LAB}.customers_current').alias('t')
     .merge(latest.alias('s'), 't.customer_id = s.customer_id')
     .whenMatchedDelete(condition="s.op = 'd'")
     # 2) out-of-order guard: never let an older event overwrite newer state
     .whenMatchedUpdateAll(condition="s.op != 'd' AND s.op_ts > t.op_ts")
     .whenNotMatchedInsertAll(condition="s.op != 'd'")
     .execute())


def run_cdc_stream():
    (spark.readStream.table(f'{LAB}.customer_changes')
     .writeStream
     .foreachBatch(upsert_customers)
     .option('checkpointLocation', f'{LAB_DIR}/_checkpoints/customers_current')
     .trigger(availableNow=True)
     .start()
     .awaitTermination())


run_cdc_stream()

current = spark.table(f'{LAB}.customers_current')
print(f'rows: {current.count()} (expected 950 = 1000 inserts - 50 deletes)')
print(f"late update leaked: {current.filter("city = 'LATE-SHOULD-LOSE'").count()} (expected 0)")
print(f"updates applied:    {current.filter("city = 'Warszawa-NEW'").count()} (expected 100)")
# exercise: rewrite this as DLT/Lakeflow apply_changes(..., sequence_by=op_ts,
#           stored_as_scd_type=2) and compare - SCD2 history for free

## 6. CDF (Change Data Feed)

A **Delta Lake feature** (vs CDC = a pattern): row-level changes readable between table
versions. Typical direction: **layer -> layer** inside the lakehouse.
Metadata columns: `_change_type` (`insert` / `update_preimage` / `update_postimage` /
`delete`), `_commit_version`, `_commit_timestamp`.

Not free: UPDATE/MERGE/DELETE additionally materialise `_change_data/` files.
Works only from the moment of enabling; `VACUUM` limits how far back you can read.

In [ ]:
# produce a second wave of changes so the feed has updates AND deletes to show
more = (as_changes(base.filter('customer_id BETWEEN 101 AND 120')
                   .withColumn('city', F.lit('Krakow-NEW')), 'u', '2026-01-05 10:00:00')
        .union(as_changes(base.filter('customer_id BETWEEN 5 AND 8'), 'd', '2026-01-05 11:00:00')))
more.write.mode('append').saveAsTable(f'{LAB}.customer_changes')

run_cdc_stream()  # checkpoint -> picks up ONLY the appended rows

# read the feed (batch mode)
feed = (spark.read.format('delta')
        .option('readChangeFeed', 'true')
        .option('startingVersion', 1)
        .table(f'{LAB}.customers_current'))

display(feed.groupBy('_commit_version', '_change_type').count().orderBy('_commit_version'))
print('update = a PAIR of rows: preimage (before) + postimage (after):')
display(feed.filter('customer_id = 101').orderBy('_commit_version'))

In [ ]:
# CDF -> downstream: incremental 1:1 replica (the silver->gold propagation pattern)
spark.sql(f'DROP TABLE IF EXISTS {LAB}.customers_mirror')
dbutils.fs.rm(f'{LAB_DIR}/_checkpoints/customers_mirror', True)
spark.table(f'{LAB}.customers_current').filter('1=0') \
    .write.saveAsTable(f'{LAB}.customers_mirror')


def mirror_changes(batch_df, batch_id):
    latest = (batch_df
              .filter(F.col('_change_type') != 'update_preimage')   # keep insert/postimage/delete
              .withColumn('rn', F.row_number().over(
                  Window.partitionBy('customer_id').orderBy(F.col('_commit_version').desc())))
              .filter('rn = 1').drop('rn'))

    (DeltaTable.forName(spark, f'{LAB}.customers_mirror').alias('t')
     .merge(latest.alias('s'), 't.customer_id = s.customer_id')
     .whenMatchedDelete(condition="s._change_type = 'delete'")
     .whenMatchedUpdateAll(condition="s._change_type != 'delete'")
     .whenNotMatchedInsertAll(condition="s._change_type != 'delete'")
     .execute())


(spark.readStream.format('delta')
 .option('readChangeFeed', 'true')
 .table(f'{LAB}.customers_current')
 .writeStream
 .foreachBatch(mirror_changes)
 .option('checkpointLocation', f'{LAB_DIR}/_checkpoints/customers_mirror')
 .trigger(availableNow=True)
 .start()
 .awaitTermination())

src, dst = spark.table(f'{LAB}.customers_current'), spark.table(f'{LAB}.customers_mirror')
print(f'source: {src.count()} rows, mirror: {dst.count()} rows -> identical: {src.count() == dst.count()}')

## 7. Delta vs Parquet

**Parquet is a file format. Delta is a table format = Parquet files + `_delta_log/`.**
Delete the log and you are left with bare immutable files: no ACID, no time travel,
no MERGE, no consistent list of "current" files.

In [ ]:
sample = spark.table(f'{CATALOG}.dim.dim_products')

# plain parquet on the volume: just files, nothing else
sample.write.mode('overwrite').parquet(f'{LAB_DIR}/products_parquet')
print('parquet dir:', [f.name for f in dbutils.fs.ls(f'{LAB_DIR}/products_parquet')][:5])

# delta table: same parquet underneath + transaction log => history, versions, ACID
sample.write.mode('overwrite').saveAsTable(f'{LAB}.products_delta')
display(spark.sql(f'DESCRIBE HISTORY {LAB}.products_delta')
        .select('version', 'timestamp', 'operation', 'operationParameters'))

In [ ]:
# UPDATE - impossible on parquet (immutable files, rewrite everything yourself);
# on Delta it is one statement (copy-on-write or deletion vectors under the hood)
spark.sql(f"""
    UPDATE {LAB}.products_delta
    SET sale_price_net = round(sale_price_net * 1.10, 2)
    WHERE category = 'Tyres'
""")

# time travel: read the state BEFORE the update
v_now = spark.table(f'{LAB}.products_delta')
v_0 = spark.read.option('versionAsOf', 0).table(f'{LAB}.products_delta')
avg = lambda df: df.filter("category = 'Tyres'").agg(F.round(F.avg('sale_price_net'), 2)).first()[0]
print(f'avg tyre price v0: {avg(v_0)}, now: {avg(v_now)}')

# undo the mistake
spark.sql(f'RESTORE TABLE {LAB}.products_delta TO VERSION AS OF 0')
print(f'after RESTORE: {avg(spark.table(f"{LAB}.products_delta"))}')

# what would VACUUM delete? (dry run; retention protects time travel)
display(spark.sql(f'VACUUM {LAB}.products_delta RETAIN 168 HOURS DRY RUN'))

In [ ]:
# OPTIMIZE on YOUR real data - the backfill produced lots of small files (one batch
# of files per day). Small files = per-file overhead on every read.
detail = spark.sql(f'DESCRIBE DETAIL {CATALOG}.fact.fact_sales_items').first()
print(f"before: {detail['numFiles']:,} files, {detail['sizeInBytes'] / 1024**3:.2f} GB")

spark.sql(f'OPTIMIZE {CATALOG}.fact.fact_sales_items')  # bin-packing (can take a while)

detail = spark.sql(f'DESCRIBE DETAIL {CATALOG}.fact.fact_sales_items').first()
print(f"after : {detail['numFiles']:,} files, {detail['sizeInBytes'] / 1024**3:.2f} GB")

# next steps to explore on your own:
#   OPTIMIZE ... ZORDER BY (product_id)      -- co-locate for data skipping
#   ALTER TABLE ... CLUSTER BY (product_id)  -- liquid clustering (incremental)
#   re-run testing/row_counts.ipynb and compare file counts

## Wrap-up – the "what gets confused with what" map

- **CDC vs CDF** – pattern (source -> lake) vs Delta feature (layer -> layer)
- **Delta vs Parquet** – table format (log) vs file format
- **AQE skew join vs salting** – automatic shuffle-partition splitting vs manual hot-key
  break-up; salting when AQE/broadcast are not enough
- **`repartition(col)` vs round-robin** – by key = skew again; without column = even but shuffles
- **Auto Loader vs COPY INTO** – scale + schema evolution + streaming vs simple, few files
- **directory listing vs file notification** – read-only simplicity vs scale + extra IAM
- **SCD1 vs SCD2** – current state vs history
- **Z-Order vs liquid clustering** – one-off sort at OPTIMIZE vs incremental
- **copy-on-write vs deletion vectors** – rewrite files vs merge-on-read delete vector

In [ ]:
# cleanup - uncomment when you are done with ALL sections
# spark.sql(f'DROP SCHEMA {LAB} CASCADE')   # drops lab tables AND the files volume
print(f'lab objects kept in {LAB} - drop the schema when done')